# 2D DAPI Section Registration

Registers a set of 2D DAPI-stained tissue section images to one chosen reference section
using **SyN deformable registration** (ANTsPy). Traced tissue outlines and cell centroids
for each section are then warped into the reference section's coordinate space.

**Expected input layout (`IMAGE_DIR`):**
```
IMAGE_DIR/
    example_section_01.tif
    example_section_02.tif
    ...
    output/                                # optional, only needed for Step 5
        example_section_01_outline_xy.csv  # traced tissue outline, x/y in microns
        example_section_01_centroids.csv   # cell centroids, x/y in microns
        ...
```


In [ ]:
from pathlib import Path
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
import ants
from scipy import ndimage as ndi

# =============================================================================
# CONFIG -- edit these for your dataset
# =============================================================================

# Folder containing the TIFF sections to register (see expected layout above).
IMAGE_DIR = Path(r"C:\path\to\your\dataset\example_group")
TIFF_GLOB = "*.tif"

# Filename stem (no extension) of the section to use as the fixed reference,
# e.g. "example_section_02". Must match one of the files matched by TIFF_GLOB.
REFERENCE_NAME = "example_section_02"

# Pixel size of the input images, in microns/pixel -- set to your imaging system's calibration.
PIXEL_SIZE_UM = 1 / 1.6028   # e.g. 0.624 um/px at 1.6028 px/um

# --- Preprocessing ---
BORDER_PAD = 100   # Zero-pad each image before cropping so the tissue never sits flush
                    # against the registration canvas edge; prevents top/edge cut-offs.
BLUR_SIGMA = 8.0    # Gaussian blur before registration -- higher = ignore finer features.

# --- Registration ---
TRANSFORM_TYPE = "SyNRA"   # SyNRA = Rigid -> Affine -> SyN; handles position/scale
                            # differences between sections before deformable warp.
                            # Use "SyN" if all sections are already well-centred and
                            # similarly sized.
GRAD_STEP = 0.1
FLOW_SIGMA = 6.0            # Smoothness of deformation field; higher = less local distortion.
TOTAL_SIGMA = 0.5           # ANTsPy default; 0 = no extra global regularization.
REG_ITERATIONS = (100, 70, 20, 0)   # 0 at finest scale = no full-res SyN deformation.
SYN_METRIC = "MI"           # MI = Mutual Information; robust to intensity differences
                             # between sections. Use "CC" if sections have very similar
                             # intensity patterns.

# Internal registration canvas size -- sections are cropped to their bounding box and
# resampled to this shape before registration.
TARGET_SHAPE = (512, 512)
PADDING = 60   # buffer around the tissue bounding box in original pixels; governs
               # headroom in the registration canvas for rigid-body translation.

WARPED_OUTPUT = IMAGE_DIR / "warped_output"
WARPED_OUTPUT.mkdir(parents=True, exist_ok=True)


## 1. Load images

In [ ]:
image_paths = sorted(IMAGE_DIR.glob(TIFF_GLOB))
assert len(image_paths) >= 2, "Need at least 2 images."

# Resolve reference by name
stems = [p.stem for p in image_paths]
assert REFERENCE_NAME in stems, f"REFERENCE_NAME '{REFERENCE_NAME}' not found. Available: {stems}"
REFERENCE_INDEX = stems.index(REFERENCE_NAME)

print(f"Found {len(image_paths)} images. Reference: {image_paths[REFERENCE_INDEX].name}")

def load_as_ants(path: Path) -> ants.ANTsImage:
    arr = tifffile.imread(str(path))
    if arr.ndim == 3:
        arr = arr.mean(axis=-1)
    return ants.from_numpy(arr.astype(np.float32))

images = [load_as_ants(p) for p in image_paths]

n = len(images)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
for ax, img, path in zip(np.array(axes).flatten(), images, image_paths):
    ax.imshow(img.numpy(), cmap="gray")
    label = f"[REFERENCE] {path.stem}" if path == image_paths[REFERENCE_INDEX] else path.stem
    ax.set_title(label, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 2. Preprocess -- crop, resample, normalize

Each section is cropped to its bounding box and resampled to 512x512 for registration.
A Gaussian blur is applied to the normalized images only -- original intensities are preserved for output.

In [ ]:
def crop_and_resample(img: ants.ANTsImage):
    """Crop to tissue bounding box, resample to TARGET_SHAPE, reset spacing to (1,1).

    The image is zero-padded by BORDER_PAD on all sides before bounding-box detection.
    This guarantees buffer space even when the tissue extends to the TIFF edge, preventing
    content from being clipped off during registration warping.
    Returned crop coordinates are in original (unpadded) image space and may be negative
    when the tissue was flush against an edge -- this is handled correctly by the coordinate
    mapping in Step 5 and the native crop save in Step 3.
    """
    arr = img.numpy()

    # Pad so the tissue is never flush against the canvas boundary
    arr_pad = np.pad(arr, BORDER_PAD, mode='constant', constant_values=0)

    nonzero = arr_pad[arr_pad > 0]
    thresh = np.percentile(nonzero, 5) if len(nonzero) > 0 else 0
    mask = arr_pad > thresh

    close_px = max(10, int(min(arr_pad.shape) * 0.02))
    mask = ndi.binary_closing(mask, structure=np.ones((close_px, close_px)))

    rows_any, cols_any = np.any(mask, axis=1), np.any(mask, axis=0)
    r0 = max(0, np.argmax(rows_any) - PADDING)
    r1 = min(arr_pad.shape[0], len(rows_any) - np.argmax(rows_any[::-1]) + PADDING)
    c0 = max(0, np.argmax(cols_any) - PADDING)
    c1 = min(arr_pad.shape[1], len(cols_any) - np.argmax(cols_any[::-1]) + PADDING)
    cropped = arr_pad[r0:r1, c0:c1].astype(np.float32)
    crop_h, crop_w = cropped.shape

    resampled = ants.resample_image(
        ants.from_numpy(cropped), TARGET_SHAPE, use_voxels=True, interp_type=4
    )

    # Convert crop coords back to original (unpadded) image space
    return (ants.from_numpy(resampled.numpy().astype(np.float32)),
            (r0 - BORDER_PAD, r1 - BORDER_PAD, c0 - BORDER_PAD, c1 - BORDER_PAD),
            (crop_h, crop_w))


def normalize_and_blur(img: ants.ANTsImage, sigma: float) -> ants.ANTsImage:
    """N4 + normalize to [0,1] + Gaussian blur. Used for registration only."""
    corrected = ants.n4_bias_field_correction(
        img, shrink_factor=2,
        convergence={"iters": [50, 50, 50], "tol": 1e-6}, verbose=False,
    )
    arr = np.clip(corrected.numpy(), 0, np.percentile(corrected.numpy(), 95))
    lo, hi = arr.min(), arr.max()
    arr = (arr - lo) / (hi - lo + 1e-8)
    arr = ndi.gaussian_filter(arr, sigma=sigma)
    return ants.from_numpy(arr.astype(np.float32))


results        = [crop_and_resample(img) for img in images]
images_cropped = [r[0] for r in results]
crop_coords    = [r[1] for r in results]
crop_shapes    = [r[2] for r in results]

images_norm = [normalize_and_blur(img, BLUR_SIGMA) for img in images_cropped]

print(f"All images cropped (BORDER_PAD={BORDER_PAD}, PADDING={PADDING}), resampled to {TARGET_SHAPE}, normalized and blurred (sigma={BLUR_SIGMA}).")
print("Crop sizes at original pixel scale:")
for path, (r0,r1,c0,c1), (ch,cw) in zip(image_paths, crop_coords, crop_shapes):
    print(f"  {path.stem}: {ch} x {cw} px  (crop origin: r0={r0}, c0={c0})")


In [ ]:
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
for ax, img, path in zip(np.array(axes).flatten(), images_norm, image_paths):
    arr = img.numpy()
    ax.imshow(arr, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{path.stem}\nmin={arr.min():.2f}  max={arr.max():.2f}", fontsize=8)
    ax.axis("off")
plt.suptitle("Normalized images -- all at 512x512, tissue filling canvas", fontsize=12)
plt.tight_layout()
plt.show()


## 3. Register all sections to the reference

SyN deformable registration on Gaussian-blurred normalized images.
Warped TIFFs are saved at the reference section's native crop resolution.

In [ ]:
fixed_norm = images_norm[REFERENCE_INDEX]
fixed_orig = images_cropped[REFERENCE_INDEX]
fixed_path = image_paths[REFERENCE_INDEX]

# Native reference crop dimensions -- all saved TIFFs will match this size
ref_crop_h_save, ref_crop_w_save = crop_shapes[REFERENCE_INDEX]
ref_r0s, ref_r1s, ref_c0s, ref_c1s = crop_coords[REFERENCE_INDEX]

print("=== Registration parameters ===")
print(f"  Reference:       {fixed_path.stem}")
print(f"  TRANSFORM_TYPE:  {TRANSFORM_TYPE}")
print(f"  SYN_METRIC:      {SYN_METRIC}")
print(f"  REG_ITERATIONS:  {REG_ITERATIONS}")
print(f"  GRAD_STEP:       {GRAD_STEP}")
print(f"  FLOW_SIGMA:      {FLOW_SIGMA}")
print(f"  TOTAL_SIGMA:     {TOTAL_SIGMA}")
print(f"  BLUR_SIGMA:      {BLUR_SIGMA}")
print(f"  BORDER_PAD:      {BORDER_PAD}")
print(f"  PIXEL_SIZE_UM:   {PIXEL_SIZE_UM:.7f}")
print(f"  masking:         personalized fixed image per section")
print("=" * 33)

registrations = {}
warped_images = {}

warped_dir = WARPED_OUTPUT / "warped"
warped_dir.mkdir(exist_ok=True)

for img_norm, img_orig, path, img_raw in zip(images_norm, images_cropped, image_paths, images):
    if path == fixed_path:
        warped_images[path.stem] = img_orig
        raw_arr = img_raw.numpy()
        h_raw, w_raw = raw_arr.shape

        r0_clip = max(0, ref_r0s);  r1_clip = min(h_raw, ref_r1s)
        c0_clip = max(0, ref_c0s);  c1_clip = min(w_raw, ref_c1s)

        native_crop = raw_arr[r0_clip:r1_clip, c0_clip:c1_clip]

        top_pad    = max(0, -ref_r0s)
        bottom_pad = max(0, ref_r1s - h_raw)
        left_pad   = max(0, -ref_c0s)
        right_pad  = max(0, ref_c1s - w_raw)
        if any([top_pad, bottom_pad, left_pad, right_pad]):
            native_crop = np.pad(native_crop,
                                 ((top_pad, bottom_pad), (left_pad, right_pad)),
                                 'constant')

        out = np.clip(native_crop, 0, 65535).astype(np.uint16)
        tifffile.imwrite(str(warped_dir / f"registered_{path.stem}.tif"), out)
        print(f"Reference (no transform): {path.stem}  [{ref_crop_w_save}x{ref_crop_h_save} px]")
        continue

    img_norm_matched = ants.histogram_match_image(img_norm, fixed_norm)

 
    orig_arr = img_orig.numpy()
    orig_nonzero = orig_arr[orig_arr > 0]
    orig_thresh = np.percentile(orig_nonzero, 5) if len(orig_nonzero) > 0 else 0
    moving_tissue_mask = ndi.binary_closing(
        orig_arr > orig_thresh, structure=np.ones((20, 20))
    )
    fixed_personalized = ants.from_numpy(
        (fixed_norm.numpy() * moving_tissue_mask).astype(np.float32)
    )

    reg = ants.registration(
        fixed=fixed_personalized,
        moving=img_norm_matched,
        type_of_transform=TRANSFORM_TYPE,
        grad_step=GRAD_STEP,
        flow_sigma=FLOW_SIGMA,
        total_sigma=TOTAL_SIGMA,
        syn_metric=SYN_METRIC,
        syn_sampling=32,
        reg_iterations=REG_ITERATIONS,
        verbose=False,
    )
    registrations[path.stem] = reg

    warped = ants.apply_transforms(
        fixed=fixed_orig,
        moving=img_orig,
        transformlist=reg["fwdtransforms"],
    )
    warped_images[path.stem] = warped

    warped_native = ants.resample_image(
        warped, (ref_crop_h_save, ref_crop_w_save), use_voxels=True, interp_type=4
    )
    out = np.clip(warped_native.numpy(), 0, 65535).astype(np.uint16)
    tifffile.imwrite(str(warped_dir / f"registered_{path.stem}.tif"), out)
    print(f"Registered: {path.stem}  [{ref_crop_w_save}x{ref_crop_h_save} px]")

print(f"\nAll TIFFs saved at native reference crop size: {ref_crop_w_save}x{ref_crop_h_save} px")
print(f"Saved to: {warped_dir}")


## 4. Alignment check -- checkerboard

In [ ]:
def normalize_display(arr):
    lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)
    return np.clip((arr - lo) / (hi - lo + 1e-8), 0, 1)

def checkerboard(a, b, tile=40):
    h, w = a.shape
    mask = (np.arange(h)[:, None] // tile + np.arange(w)[None, :] // tile) % 2 == 0
    return np.where(mask, a, b)

moving_paths = [p for p in image_paths if p != fixed_path]
n_moving = len(moving_paths)
fixed_arr = normalize_display(fixed_orig.numpy())

fig, axes = plt.subplots(1, n_moving, figsize=(6 * n_moving, 6))
for ax, path in zip(np.array(axes).flatten(), moving_paths):
    warped_arr = normalize_display(warped_images[path.stem].numpy())
    cb = checkerboard(fixed_arr, warped_arr)
    ax.imshow(cb, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"reference / {path.stem}", fontsize=8)
    ax.axis("off")
plt.suptitle("Checkerboard: reference vs. registered section", fontsize=12)
plt.tight_layout()
plt.savefig(WARPED_OUTPUT / "alignment_check.png", dpi=150)
plt.show()


## 4b. Save transforms

Copies the registration transform files to a permanent location so registration does not need to be re-run.

In [ ]:
transforms_dir = WARPED_OUTPUT / "transforms"
transforms_dir.mkdir(exist_ok=True)

for stem, reg in registrations.items():
    stem_dir = transforms_dir / stem
    stem_dir.mkdir(exist_ok=True)

    saved_fwd, saved_inv = [], []

    for src in reg["fwdtransforms"]:
        dst = stem_dir / f"fwd_{Path(src).name}"
        shutil.copy2(src, dst)
        saved_fwd.append(str(dst))

    for src in reg["invtransforms"]:
        dst = stem_dir / f"inv_{Path(src).name}"
        shutil.copy2(src, dst)
        saved_inv.append(str(dst))

    # Update registrations dict to point to the saved copies
    registrations[stem]["fwdtransforms"] = saved_fwd
    registrations[stem]["invtransforms"] = saved_inv

    print(f"{stem}: saved {len(saved_fwd)} fwd + {len(saved_inv)} inv transforms -> {stem_dir}")

print("\nTransforms saved. Subsequent cells will use these permanent files.")


## 5. Warp tissue outlines and cell centroids to reference space

Reads from `IMAGE_DIR/output/`:
- `<stem>_outline_xy.csv` -- manually traced tissue outline, X/Y in microns
- `<stem>_centroids.csv` -- cell centroids, X/Y in microns

Both are warped to the reference coordinate space. Output CSV columns:
- `x`, `y` -- position in 512x512 registration space (used by the overlay plot)
- `native_x`, `native_y` -- pixel coordinates in the saved native-resolution warped TIFF
- `x_um`, `y_um` -- physical position in microns in the reference image coordinate system

Tissue outlines are processed for all images including the reference (coordinates converted,
no warp applied). Centroids are skipped for the reference image.

In [ ]:
warped_csvs_dir = WARPED_OUTPUT / "warped_csvs"
warped_csvs_dir.mkdir(exist_ok=True)

input_dir = IMAGE_DIR / "output"

target_h, target_w = TARGET_SHAPE

ref_r0, ref_r1, ref_c0, ref_c1 = crop_coords[REFERENCE_INDEX]
ref_crop_h, ref_crop_w = crop_shapes[REFERENCE_INDEX]


def load_imagej_xy(csv_path: Path) -> pd.DataFrame:
    """Load ImageJ XY CSV (outline or centroids). Converts microns -> pixels.
    Accepts column names 'x'/'y' or 'x_microns'/'y_microns'."""
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    rename = {}
    if "x_microns" in df.columns: rename["x_microns"] = "x"
    if "y_microns" in df.columns: rename["y_microns"] = "y"
    if rename:
        df = df.rename(columns=rename)
    if "x" not in df.columns or "y" not in df.columns:
        raise KeyError(f"Cannot find x/y columns in {csv_path.name}. Found: {list(df.columns)}")
    df["x"] = df["x"] / PIXEL_SIZE_UM   # microns -> col pixels
    df["y"] = df["y"] / PIXEL_SIZE_UM   # microns -> row pixels
    return df


def warp_points(df: pd.DataFrame, c0: int, r0: int,
                crop_w: int, crop_h: int,
                is_reference: bool, stem: str) -> pd.DataFrame:
    """Map ImageJ pixel coords -> reference space.
    Reference: convert coordinates only (no transform). Moving: apply invtransforms."""
    col_reg = (df["x"] - c0) / crop_w * target_w
    row_reg = (df["y"] - r0) / crop_h * target_h

    if is_reference:
        col_w = col_reg.values
        row_w = row_reg.values
    else:
        # ANTsPy physical coords for from_numpy: dim0=rows="x", dim1=cols="y"
        df_ants = pd.DataFrame({"x": row_reg, "y": col_reg})
        warped_ants = ants.apply_transforms_to_points(
            dim=2,
            points=df_ants,
            transformlist=registrations[stem]["invtransforms"],
        )
        col_w = warped_ants["y"].values   # ANTs y = col
        row_w = warped_ants["x"].values   # ANTs x = row

    x_um = (col_w * (ref_crop_w / target_w) + ref_c0) * PIXEL_SIZE_UM
    y_um = (row_w * (ref_crop_h / target_h) + ref_r0) * PIXEL_SIZE_UM

    out = pd.DataFrame({
        "x":        col_w,                            # col in 512x512 (overlay plot)
        "y":        row_w,                            # row in 512x512 (overlay plot)
        "native_x": col_w * ref_crop_w / target_w,   # col in saved native-res TIFF
        "native_y": row_w * ref_crop_h / target_h,   # row in saved native-res TIFF
        "x_um":     x_um,
        "y_um":     y_um,
    })
    for col in df.columns:
        if col not in ("x", "y"):
            out[col] = df[col].values
    return out


for i, path in enumerate(image_paths):
    r0, r1, c0, c1 = crop_coords[i]
    crop_h, crop_w  = crop_shapes[i]
    is_reference = (path == fixed_path)
    stem = path.stem

    # --- Tissue outline (all images, including reference) ---
    outline_csv = input_dir / f"{stem}_outline_xy.csv"
    if outline_csv.exists():
        df_outline = load_imagej_xy(outline_csv)
        warped_outline = warp_points(df_outline, c0, r0, crop_w, crop_h, is_reference, stem)
        warped_outline.to_csv(warped_csvs_dir / f"outline_{stem}.csv", index=False)
        tag = " [reference]" if is_reference else ""
        print(f"Outline saved: {stem}  ({len(warped_outline)} points){tag}")
    else:
        print(f"Outline CSV not found, skipping: {outline_csv.name}")

    # --- Centroids (moving images only) ---
    if is_reference:
        print(f"  Skipping centroids for reference: {stem}")
        continue

    centroid_csv = input_dir / f"{stem}_centroids.csv"
    if not centroid_csv.exists():
        print(f"  No centroid CSV found: {centroid_csv.name}")
        continue

    df_centroids = load_imagej_xy(centroid_csv)
    warped_centroids = warp_points(df_centroids, c0, r0, crop_w, crop_h,
                                   is_reference=False, stem=stem)
    warped_centroids.to_csv(warped_csvs_dir / f"registered_centroids_{stem}.csv", index=False)
    print(f"  Saved {len(warped_centroids)} centroids.")

print("\nDone.")


## 6. Overlay -- reference image + all centroid clouds

In [ ]:
colors = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(normalize_display(fixed_orig.numpy()), cmap="gray", vmin=0, vmax=1)

for idx, path in enumerate(image_paths):
    color = colors[idx % len(colors)]

    # Centroids
    csv_path = warped_csvs_dir / f"registered_centroids_{path.stem}.csv"
    if csv_path.exists():
        pts = pd.read_csv(csv_path)
        ax.scatter(pts["x"], pts["y"], s=4, alpha=0.6, color=color, label=path.stem)

    # Outlines
    out_path = warped_csvs_dir / f"outline_{path.stem}.csv"
    if out_path.exists():
        ol = pd.read_csv(out_path)
        ax.plot(ol["x"], ol["y"], lw=1, alpha=0.7, color=color)

ax.legend(fontsize=7, markerscale=3)
ax.set_title(f"All sections registered to: {fixed_path.stem}", fontsize=10)
ax.axis("off")
plt.tight_layout()
plt.savefig(WARPED_OUTPUT / "centroid_overlay.png", dpi=150)
plt.show()
